# Volatility Dynamics 15 — Student-t **return** Markov transitions

**Frozen scientific protocol:** [VD 032](../../discussions/Volatility_Dynamics/032_chatgpt_vd15_student_t_return_markov_preregistration.md). **Historical results exploratory only.**

This experiment is about a five-minute **return** transition kernel. It does **not** apply a Student-t law to stochastic-volatility-factor innovations, nor does it fit a fresh Student-t distribution to each terminal horizon.

Compared models: `T_RW` (iid native t returns), `T_MC` (native t AR(1) returns), `AM_G` (unchanged VD14 Markov Gaussian-mixture benchmark), `AM_T` (native Student-t return innovation with stochastic Markov factor paths), `AM_T_AR` (AM_T plus return Markov AR(1)), and `M2_student_t` (VD14 direct horizon-specific Student-t). Seven horizons: 5/10/15/30/60/120/240 minutes, primary 60m. Six assets with sufficient historical data.

Only local `DERIBIT_DATA_ROOT/history/price_5m` price files. No option pricing, no database writes, no cross-branch model imports, no fitted results from VD14. The historical sample has been previously inspected; output is **not independent confirmation**.

## 0. Environment and fixed registration

In [ ]:
from pathlib import Path
import os, sys, json, time, warnings
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run from the project repository.'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as base
from data_engineering.modeling import student_t_return_markov as mk
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT to your historical market-data root.'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), f'Missing price history directory: {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'15'
OUT.mkdir(parents=True,exist_ok=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
PARTICLES=int(os.getenv('VD15_PARTICLES','256'))
BOOT=int(os.getenv('VD15_BOOTSTRAPS','1000'))
assert PARTICLES==mk.PARTICLES_DEFAULT and BOOT==mk.B_DEFAULT, ('Binding historical protocol requires 256 particles and 1000 resamples. Set both defaults.')
print('VD15',mk.VERSION,'| historical exploratory only | source',HIST)
print(json.dumps(mk.registration(),indent=2,default=str))


## 1. Verify implementation before loading market data

In [ ]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
       str(ROOT/'tests'/'test_student_t_return_markov.py')],
       cwd=str(ROOT),text=True,capture_output=True)
print(p.stdout[-3500:],p.stderr[-1200:])
assert p.returncode==0, 'Synthetic scientific tests must pass before reading real data.'


## 2. Construct causal native-five-minute data panel

Only `state_time` and `close` are read from external historical CSVs. Returns are native 5m close-to-close and are set to missing at data gaps. Same de-clocking and 30m/240m causal EMA state as VD14; target `R_h` sums future returns within split and rejects missing intermediate bars. Never fit on future targets.

In [ ]:
START=time.monotonic()
raw,excluded=base.load_returns(HIST,ASSETS)
print('Excluded:',excluded,'raw observations:',len(raw))
panel=base.panelize(raw)
del raw
features=base.make_features(panel)
frame=base.make_targets(panel,features,mk.HORIZONS)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size'),
        valid_returns=('return_5m','count')).to_string())
print('Causal panel ready; seconds',round(time.monotonic()-START,1))


## 3. Run six model comparisons at all seven horizons

All six models within a horizon are evaluated at exactly the same forecast origins. Native return Student-t degrees of freedom are selected exclusively on **5m VALID** outcomes and are held fixed at longer horizons. AM-T densities integrate 5m Student-t kernels with Markov factor paths using 256 randomized-Sobol Rao-Blackwell quadrature nodes. Baseline M2 and AM-G are recomputed from the original VD14 model definitions; no output CSV is imported for model fitting.

In [ ]:
scores,boot,per_asset,parameters=mk.run_all(frame,particles=PARTICLES,B=BOOT)
assert len(scores)==len(mk.HORIZONS)*len(mk.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print('Scored in',round(time.monotonic()-START,1),'seconds')
print(scores.groupby('horizon_min').agg(models=('model','nunique'),
      origins=('n_origins','first'),days=('n_days','first')).to_string())


## 4. All-horizon scores, primary 60-minute comparisons and asset-level stability

In [ ]:
display(scores.round(7))
primary=boot[(boot.horizon_min==mk.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))]
display(primary.round(7))
display(per_asset[(per_asset.horizon_min==mk.PRIMARY)].round(7))


## 5. Numerical integration sensitivity — validation only

QMC errors can mimic a small likelihood gain. Compare 256 vs 512 nested integration nodes on a fixed, evenly spaced 60m VALID sample (up to 512 origins per asset); **do not tune the primary test decision** using numerical sensitivity.

In [ ]:
particle_check=mk.numerical_sensitivity(frame,parameters['model_kernels'],
                      parameters['volatility_markov_states'],particles=PARTICLES)
display(particle_check.round(8))
print('Numerical diagnostics are descriptive and cannot authorize post-test model changes.')


## 6. Preregistered decision: exploratory evidence only

In [ ]:
result=mk.decision(scores,boot)
print(json.dumps(result,indent=2))
assert result['status']!='CONFIRMED'
assert not result['historical_confirmation_permitted']


## 7. Save compact scored outputs, model parameters and complete manifest

In [ ]:
scores.to_csv(OUT/'15_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'15_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'15_per_asset.csv',index=False,float_format='%.10g')
particle_check.to_csv(OUT/'15_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'15_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_15','version':mk.VERSION,
          'registration':mk.registration(),'history_dir':str(HIST),
          'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
          'sample_start':str(panel.state_time.min()),'sample_end':str(panel.state_time.max()),
          'split_days':panel.groupby('split').date.nunique().to_dict(),
          'reference_implementation':'VD14 original definitions refit on same splits',
          'parameters':parameters,'particle_sensitivity_validation_only':True,
          'decision':result,'scientific_caveat':'Previously viewed two-year historical data; no prospective claim.'}
(OUT/'15_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved six compact artifacts in',OUT)
print('No origin-level return data, option quotes or DuckDB tables written.')
